In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Job Hunt Crew

## What We're Building

A job-application preparation crew with four specialists:

| Agent | Role |
|-------|------|
| JD Analyzer | Parse job descriptions and extract true hiring priorities |
| Resume Tailor | Adapt resume language and ordering to match the role honestly |
| Cover-Letter Writer | Draft a targeted cover letter tied to JD + resume context |
| Interview Coach | Build interview prep aligned to role signals and candidate background |

## Crew Task Handoff

1. JD analysis sets requirement priorities and hidden signals.
2. Resume tailoring uses those priorities to reframe experience.
3. Cover-letter writing uses JD analysis plus tailored resume context.
4. Interview coaching uses outputs from earlier steps to generate realistic prep.

## Learning Notes on Role Design

- Keep each role narrow: one role = one decision boundary.
- Place analysis roles before generation roles to reduce hallucination.
- Give each downstream role explicit context from upstream outputs.
- Add one quality role (here: interview coach) that stress-tests prior artifacts.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Job Description & Resume

In [3]:
job_description = """
Role: Senior Machine Learning Engineer
Company: HealthAI Inc. (Series B, $50M raised)
Location: Remote (US)
Salary: $180,000 - $220,000 + equity

About us: HealthAI builds AI-powered diagnostic tools for hospitals.
We process millions of medical images daily using deep learning.

Requirements:
- 5+ years ML engineering experience
- Strong Python (PyTorch preferred)
- Experience with computer vision / image classification
- MLOps: model deployment, monitoring, CI/CD for ML
- Experience with cloud (AWS or GCP)
- Healthcare/regulated industry experience is a plus
- Published papers or patents is a bonus

Nice-to-have:
- Experience with HIPAA compliance
- Knowledge of medical imaging (DICOM, radiology)
- Team lead experience

What you'll do:
- Lead development of image classification models for radiology
- Build and maintain ML pipelines (training, evaluation, deployment)
- Collaborate with clinicians to validate model performance
- Mentor junior engineers
"""

candidate_resume = """
Name: Alex Chen
Current Role: ML Engineer at TechCorp (3 years)
Previous: Data Scientist at RetailAI (2 years)
Education: MS Computer Science, Stanford University

Skills:
- Python, PyTorch, TensorFlow, scikit-learn
- Computer Vision: image classification, object detection, segmentation
- MLOps: MLflow, Docker, Kubernetes, AWS SageMaker
- Cloud: AWS (3 years), GCP (1 year)
- Languages: Python, SQL, Bash

Experience highlights:
- Built a product defect detection system (98.5% accuracy, $2M/year savings)
- Deployed 15+ models to production using MLflow + K8s
- Led a team of 3 junior engineers
- Published 2 papers at CVPR on efficient architecture search

Missing from resume:
- No healthcare industry experience
- No HIPAA experience
- No medical imaging (DICOM) experience
"""

print("JD and resume loaded")

JD and resume loaded


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    jd_analyzer = Agent(
        role="JD Analyzer",
        goal="Parse job descriptions to extract requirements, priorities, and hidden signals",
        backstory=(
            "You are a career coach who has reviewed thousands of JDs and can separate"
            " true requirements from generic posting language."
        ),
        llm=llm,
        verbose=True,
    )

    resume_tailor = Agent(
        role="Resume Tailor",
        goal="Tailor the candidate resume to maximize role match without fabrication",
        backstory=(
            "You are a resume strategist focused on truthful optimization, ATS readability,"
            " and impact-forward bullet rewriting."
        ),
        llm=llm,
        verbose=True,
    )

    cover_letter_writer = Agent(
        role="Cover-Letter Writer",
        goal="Write a targeted cover letter that closes perceived fit gaps",
        backstory=(
            "You write concise, evidence-based cover letters that tie candidate outcomes"
            " to team needs in plain language."
        ),
        llm=llm,
        verbose=True,
    )

    interview_coach = Agent(
        role="Interview Coach",
        goal="Prepare realistic interview Q&A using JD + tailored application outputs",
        backstory=(
            "You are a senior hiring manager who converts role requirements into practical"
            " interview drills and strong STAR-style responses."
        ),
        llm=llm,
        verbose=True,
        allow_delegation=True,
    )
else:
    jd_analyzer = {"role": "JD Analyzer"}
    resume_tailor = {"role": "Resume Tailor"}
    cover_letter_writer = {"role": "Cover-Letter Writer"}
    interview_coach = {"role": "Interview Coach"}

print("4 agents created: JD Analyzer, Resume Tailor, Cover-Letter Writer, Interview Coach")


4 agents created: JD Analyzer, Resume Tailor, Cover-Letter Writer, Interview Coach


## Step 4 — Create Tasks

In [5]:
if CREWAI_AVAILABLE:
    analyze_task = Task(
        description=f"""Analyze this job description:

{job_description}

Return:
1. Core required skills and nice-to-haves
2. Priority signals (what matters most)
3. Hidden signals about team context and expectations""",
        expected_output="Structured JD analysis with priorities and hidden signals.",
        agent=jd_analyzer,
    )

    resume_task = Task(
        description=f"""Tailor this resume to the analyzed JD context:

{candidate_resume}

Rules:
1. Reorder and rewrite bullets for strongest relevance
2. Add only truthful keyword alignment
3. Keep concise, impact-first formatting""",
        expected_output="Tailored resume draft aligned to JD priorities.",
        agent=resume_tailor,
    )

    cover_letter_task = Task(
        description="""Write a tailored cover letter using JD analysis and resume updates.

Requirements:
1. Tie candidate strengths to role needs
2. Address one likely fit concern directly
3. Keep under 400 words and action-oriented""",
        expected_output="Personalized cover letter draft under 400 words.",
        agent=cover_letter_writer,
        context=[analyze_task, resume_task],
    )

    interview_task = Task(
        description="""Create interview prep based on all prior outputs.

Deliver:
1. 8 likely interview questions
2. STAR-style answer outlines
3. 3 questions candidate should ask interviewer
4. 3 risk areas and mitigation tactics""",
        expected_output="Interview prep guide with questions, answers, and strategy notes.",
        agent=interview_coach,
        context=[analyze_task, resume_task, cover_letter_task],
    )
else:
    analyze_task = {"name": "JD analysis task"}
    resume_task = {"name": "Resume tailoring task"}
    cover_letter_task = {"name": "Cover-letter task"}
    interview_task = {"name": "Interview coaching task"}

print("4 tasks created with handoff: JD analysis -> resume tailoring -> cover letter -> interview coaching")


4 tasks created with handoff: JD analysis -> resume tailoring -> cover letter -> interview coaching


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Interview prep package generated from JD, tailored resume, and cover letter context."
        self.tasks_output = [
            _DemoTaskOutput("JD Analysis", "Priority skills and hidden role signals extracted from JD."),
            _DemoTaskOutput("Tailored Resume", "Resume reordered and rewritten for role alignment."),
            _DemoTaskOutput("Cover Letter", "Targeted cover letter addressing fit and motivation."),
            _DemoTaskOutput("Interview Prep", "Likely questions, STAR outlines, and risk-mitigation prep."),
        ]


if CREWAI_AVAILABLE:
    job_crew = Crew(
        agents=[jd_analyzer, resume_tailor, cover_letter_writer, interview_coach],
        tasks=[analyze_task, resume_task, cover_letter_task, interview_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Job hunt crew assembled. Starting pipeline...")
    result = job_crew.kickoff()
    print("\nJob hunt workflow complete.")
else:
    print("Demo mode: showing role handoff without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing role handoff without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("INTERVIEW PREP GUIDE")
print("=" * 60)
print(result.raw)

preview("JD Analysis", result.tasks_output[0].raw)
preview("Tailored Resume", result.tasks_output[1].raw)
preview("Cover Letter", result.tasks_output[2].raw)
preview("Interview Prep", result.tasks_output[3].raw)


INTERVIEW PREP GUIDE
Interview prep package generated from JD, tailored resume, and cover letter context.

JD Analysis
Priority skills and hidden role signals extracted from JD.

Tailored Resume
Resume reordered and rewritten for role alignment.

Cover Letter
Targeted cover letter addressing fit and motivation.

Interview Prep
Likely questions, STAR outlines, and risk-mitigation prep.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Role specialization | JD Analyzer, Resume Tailor, Cover-Letter Writer, Interview Coach |
| Handoff chain | JD -> Resume -> Cover Letter -> Interview Prep |
| Explicit context wiring | Downstream tasks receive upstream artifacts via `context=[...]` |
| Learning from role design | Analysis-first design improves output quality of writing agents |

## Role Design Learning Notes

- Separate interpretation (JD Analyzer) from rewriting (Resume Tailor).
- Keep persuasion artifacts (Cover Letter) after factual alignment artifacts (Resume).
- Position Interview Coach last so it can stress-test all prior outputs together.
- Use delegation sparingly and only where uncertainty benefits from upstream clarification.
